## 1. Import Required Libraries and Setup

In [ ]:
// Key libraries and types needed for NotebookLM features:

// Frontend (React/TypeScript):
types DocumentMetadata = {
  documentId: string;
  fileName: string;
  pageNumber?: number;
  sectionTitle?: string;
  chunkIndex: number;
};

types Citation = {
  text: string;
  documentName: string;
  pageNumber: number;
  sectionTitle?: string;
  confidence: number;
};

types ChatResponse = {
  answer: string;
  citations: Citation[];
  sources: string[];
  relatedQuestions?: string[];
};

// Backend/Services:
- LangChain for document processing and RAG
- Gemini API for embeddings and generation
- Vector database (Pinecone/Weaviate) for similarity search
- pdfjs-dist for PDF text extraction with page tracking
- Natural for text processing and chunk optimization

## 2. Document Parser with Page Number Tracking

Extract text from PDFs while preserving page numbers and section structure:

In [ ]:
// DocumentParserWithMetadata.ts
export interface DocumentChunk {
  content: string;
  pageNumber: number;
  sectionTitle?: string;
  documentId: string;
  chunkIndex: number;
}

export class DocumentParserWithMetadata {
  static async parsePDFWithPages(
    file: File
  ): Promise<DocumentChunk[]> {
    const pdfDoc = await pdfjsLib.getDocument(
      await file.arrayBuffer()
    ).promise;
    const chunks: DocumentChunk[] = [];
    let chunkIndex = 0;

    for (let pageNum = 1; pageNum <= pdfDoc.numPages; pageNum++) {
      const page = await pdfDoc.getPage(pageNum);
      const textContent = await page.getTextContent();
      const pageText = textContent.items
        .map((item: any) => item.str)
        .join(" ");

      // Split by sections if headers detected
      const sections = this.extractSections(pageText);
      for (const section of sections) {
        chunks.push({
          content: section.text,
          pageNumber: pageNum,
          sectionTitle: section.title,
          documentId: file.name,
          chunkIndex: chunkIndex++
        });
      }
    }
    return chunks;
  }

  private static extractSections(
    text: string
  ): { title?: string; text: string }[] {
    // Split by common headers (##, ###, etc.)
    const headerRegex = /^(#+\s+.+)$/gm;
    const parts = text.split(headerRegex);
    const sections = [];

    for (let i = 0; i < parts.length; i += 2) {
      const title = parts[i]?.trim();
      const content = parts[i + 1]?.trim() || "";
      if (content) {
        sections.push({ title, text: content });
      }
    }
    return sections.length > 0 ? sections : [{ text }];
  }
}

## 3. Source Citation System

Format beautifulcitations like NotebookLM: "Theo trang 12 của tài liệu:..."

In [ ]:
// CitationService.ts
export class CitationService {
  /**
   * Extract citations from retrieved chunks
   * Format: "Theo trang X của tài liệu Y: "...đoạn trích dẫn...""
   */
  static formatCitations(chunks: DocumentChunk[]): string {
    return chunks
      .map((chunk, idx) => {
        const docName = chunk.documentId.replace(/\.[^.]+$/, "");
        const pageRef = chunk.pageNumber ? `trang ${chunk.pageNumber}` : "tài liệu";
        const section = chunk.sectionTitle 
          ? ` (${chunk.sectionTitle})`
          : "";

        // Get first 150 chars of content
        const excerpt = chunk.content.substring(0, 150).trim();
        
        return `**📄 ${idx + 1}. Theo ${pageRef} của \`${docName}\`${section}:**\n\n> "${excerpt}..."`;
      })
      .join("\n\n");
  }

  /**
   * Highlight citations in AI response
   * Make them interactive for mobile viewing
   */
  static enrichResponseWithCitations(
    response: string,
    chunks: DocumentChunk[]
  ): ChatResponse {
    const citations: Citation[] = chunks.map(chunk => ({
      text: chunk.content.substring(0, 200),
      documentName: chunk.documentId,
      pageNumber: chunk.pageNumber,
      sectionTitle: chunk.sectionTitle,
      confidence: 0.95
    }));

    return {
      answer: response,
      citations,
      sources: [...new Set(chunks.map(c => c.documentId))]
    };
  }

  /**
   * Format for display in chat UI
   */
  static renderCitationBubble(citation: Citation): string {
    return `
<div class="citation-bubble">
  <div class="citation-header">
    📄 ${citation.documentName}
    <span class="page-badge">trang ${citation.pageNumber}</span>
  </div>
  <div class="citation-text">"${citation.text}..."</div>
</div>
    `.trim();
  }
}

## 4. Multi-Document Reasoning

Compare documents A & B by retrieving context from both:

In [ ]:
// MultiDocumentReasoningService.ts
export class MultiDocumentReasoningService {
  /**
   * Compare multiple documents on a specific topic
   * Retrieves context from each document independently
   */
  static async compareDocuments(
    query: string,
    documentIds: string[],
    vectorStore: VectorStore
  ): Promise<ChatResponse> {
    const contextByDoc = new Map<string, DocumentChunk[]>();

    // Retrieve context from each document separately
    for (const docId of documentIds) {
      const results = await vectorStore.search(query, {
        filter: { documentId: docId },
        topK: 5
      });
      contextByDoc.set(docId, results);
    }

    // Build prompt for comparison
    const comparisonPrompt = this.buildComparisonPrompt(
      query,
      contextByDoc
    );

    // Get AI comparison
    const response = await this.queryGeminiForComparison(
      comparisonPrompt
    );

    // Collect all relevant chunks for citations
    const allChunks = Array.from(contextByDoc.values()).flat();

    return CitationService.enrichResponseWithCitations(
      response,
      allChunks
    );
  }

  private static buildComparisonPrompt(
    query: string,
    contextByDoc: Map<string, DocumentChunk[]>
  ): string {
    let prompt = `So sánh các tài liệu về: "${query}"\n\n`;

    for (const [docId, chunks] of contextByDoc) {
      prompt += `## Tài liệu: ${docId}\n`;
      prompt += chunks.map(c => c.content).join("\n");
      prompt += "\n\n";
    }

    prompt += "Hãy so sánh các tài liệu này. Chỉ ra điểm giống nhau, khác nhau, và những insight riêng.";
    return prompt;
  }

  private static async queryGeminiForComparison(
    prompt: string
  ): Promise<string> {
    // Use Gemini API
    const model = genai.getGenerativeModel({ model: "gemini-1.5-flash" });
    const result = await model.generateContent(prompt);
    return result.response.text();
  }
}

## 5. Smart Summarization

Summarize full document, specific section, or topic:

In [ ]:
// SmartSummarizationService.ts
export type SummaryType = "full" | "section" | "topic";

export class SmartSummarizationService {
  /**
   * Summarize full document
   */
  static async summarizeFullDocument(
    documentId: string,
    vectorStore: VectorStore,
    length: "short" | "medium" | "long" = "medium"
  ): Promise<string> {
    // Retrieve all chunks from document
    const allChunks = await vectorStore.search("", {
      filter: { documentId },
      topK: 100
    });

    const fullText = allChunks
      .sort((a, b) => a.chunkIndex - b.chunkIndex)
      .map(c => c.content)
      .join("\n");

    return this.generateSummary(fullText, length);
  }

  /**
   * Summarize specific section
   */
  static async summarizeSection(
    documentId: string,
    sectionTitle: string,
    vectorStore: VectorStore
  ): Promise<string> {
    const chunks = await vectorStore.search(sectionTitle, {
      filter: { 
        documentId,
        sectionTitle // Only this section
      },
      topK: 20
    });

    const sectionText = chunks.map(c => c.content).join("\n");
    return this.generateSummary(sectionText, "medium");
  }

  /**
   * Summarize topic across documents
   */
  static async summarizeTopicAcrossDocuments(
    topic: string,
    documentIds: string[],
    vectorStore: VectorStore
  ): Promise<string> {
    const topicChunks = await vectorStore.search(topic, {
      topK: 15
    });

    // Filter to only selected documents
    const filtered = topicChunks.filter(c =>
      documentIds.includes(c.documentId)
    );

    const topicText = filtered.map(c => c.content).join("\n");
    return this.generateSummary(topicText, "medium");
  }

  private static async generateSummary(
    text: string,
    length: "short" | "medium" | "long"
  ): Promise<string> {
    const lengthGuide = {
      short: "300 từ",
      medium: "500 từ",
      long: "1000 từ"
    };

    const prompt = \`
Tóm tắt đoạn văn bản sau trong khoảng \${lengthGuide[length]}:

\${text}

Tóm tắt phải:
- Bao gồm các ý chính
- Giữ lại chi tiết quan trọng
- Dễ hiểu
    \`;

    const model = genai.getGenerativeModel({ model: "gemini-1.5-flash" });
    const result = await model.generateContent(prompt);
    return result.response.text();
  }
}

## 6. Highlight & Explain Feature

Click text → Ask AI to explain with full context:

In [ ]:
// HighlightExplainService.ts
export class HighlightExplainService {
  /**
   * Explain highlighted text with context from document
   */
  static async explainHighlight(
    selectedText: string,
    documentId: string,
    context: string, // surrounding text
    vectorStore: VectorStore
  ): Promise<ChatResponse> {
    // Find the highlighted text in the document
    const contextChunks = await vectorStore.search(selectedText, {
      filter: { documentId },
      topK: 3
    });

    // Build explanation prompt with context
    const explanationPrompt = \`
User đã highlight đoạn text này:
"\${selectedText}"

Surrounding context:
\${context}

Hãy giải thích:
1. Ý nghĩa của đoạn text này
2. Tại sao nó quan trọng
3. Mối liên hệ với phần còn lại của tài liệu
4. Ứng dụng thực tế nếu có

Be clear, detailed, và dễ hiểu cho người không chuyên.
    \`;

    const model = genai.getGenerativeModel({ model: "gemini-1.5-flash" });
    const result = await model.generateContent(explanationPrompt);
    const explanation = result.response.text();

    return CitationService.enrichResponseWithCitations(
      explanation,
      contextChunks
    );
  }

  /**
   * React component for text highlighting
   */
  static renderHighlightableText(
    content: string,
    onHighlight: (selectedText: string, context: string) => void
  ): JSX.Element {
    return (
      <div 
        onMouseUp={() => {
          const selection = window.getSelection();
          if (selection && selection.toString()) {
            const selectedText = selection.toString();
            // Get surrounding context
            const range = selection.getRangeAt(0);
            const preContext = range.getBoundingClientRect();
            
            onHighlight(selectedText, content);
          }
        }}
        className="highlightable-text select-text hover:bg-yellow-100"
      >
        {content}
      </div>
    );
  }
}

## 7. Beautiful Response Formatting UI

Display AI responses professionally like NotebookLM:

In [ ]:
// ChatResponseDisplay.tsx
import React from "react";
import { Citation, ChatResponse } from "../types";

export const ChatResponseDisplay: React.FC<{
  response: ChatResponse;
}> = ({ response }) => {
  const [expandedCitations, setExpandedCitations] = React.useState<
    Set<number>
  >(new Set());

  const toggleCitation = (index: number) => {
    const newSet = new Set(expandedCitations);
    if (newSet.has(index)) {
      newSet.delete(index);
    } else {
      newSet.add(index);
    }
    setExpandedCitations(newSet);
  };

  return (
    <div className="chat-response-container space-y-6 p-4">
      {/* Main Answer */}
      <div className="answer-section">
        <div className="prose prose-sm max-w-full">
          <div className="whitespace-pre-wrap text-gray-800 leading-relaxed">
            {response.answer}
          </div>
        </div>
      </div>

      {/* Citations Section */}
      {response.citations.length > 0 && (
        <div className="citations-section border-t pt-4">
          <h3 className="text-sm font-semibold text-gray-700 mb-3">
            📚 Trích dẫn nguồn ({response.citations.length})
          </h3>

          <div className="space-y-2">
            {response.citations.map((citation, idx) => (
              <div
                key={idx}
                className="citation-item border-l-4 border-blue-400 bg-blue-50 p-3 rounded cursor-pointer hover:bg-blue-100 transition"
                onClick={() => toggleCitation(idx)}
              >
                {/* Citation Header */}
                <div className="flex items-start justify-between">
                  <div className="flex-1">
                    <p className="font-medium text-sm text-blue-900">
                      📄 {citation.documentName}
                      {citation.sectionTitle && (
                        <span className="text-xs text-blue-700 ml-2">
                          → {citation.sectionTitle}
                        </span>
                      )}
                    </p>
                    <p className="text-xs text-blue-600 mt-1">
                      Trang {citation.pageNumber}
                    </p>
                  </div>
                  <span className="text-xl">
                    {expandedCitations.has(idx) ? "▼" : "▶"}
                  </span>
                </div>

                {/* Citation Text (Expandable) */}
                {expandedCitations.has(idx) && (
                  <div className="mt-3 p-2 bg-white border-l-2 border-blue-300 rounded">
                    <p className="text-sm text-gray-700 italic">
                      "{citation.text}..."
                    </p>
                    <div className="flex gap-2 mt-2">
                      <button className="text-xs px-2 py-1 bg-blue-200 hover:bg-blue-300 rounded">
                        Copy
                      </button>
                      <button className="text-xs px-2 py-1 bg-amber-200 hover:bg-amber-300 rounded">
                        Highlight
                      </button>
                    </div>
                  </div>
                )}
              </div>
            ))}
          </div>
        </div>
      )}

      {/* Sources Summary */}
      {response.sources.length > 0 && (
        <div className="sources-section border-t pt-3">
          <p className="text-xs text-gray-600">
            <span className="font-semibold">Từ các tài liệu:</span>{" "}
            {response.sources.join(", ")}
          </p>
        </div>
      )}

      {/* Related Questions */}
      {response.relatedQuestions && response.relatedQuestions.length > 0 && (
        <div className="related-questions border-t pt-4">
          <p className="text-sm font-semibold text-gray-700 mb-2">
            💡 Câu hỏi liên quan:
          </p>
          <div className="space-y-2">
            {response.relatedQuestions.map((q, idx) => (
              <button
                key={idx}
                className="w-full text-left text-sm p-2 bg-gradient-to-r from-indigo-50 to-purple-50 border border-indigo-200 rounded hover:border-indigo-400 hover:bg-indigo-100 transition"
              >
                {q}
              </button>
            ))}
          </div>
        </div>
      )}
    </div>
  );
};

## 8. Implementation Checklist

Files to create/update:

### New Services
- ✅ `src/services/CitationService.ts` - Extract & format citations
- ✅ `src/services/MultiDocumentReasoningService.ts` - Compare documents
- ✅ `src/services/SmartSummarizationService.ts` - Flexible summaries
- ✅ `src/services/HighlightExplainService.ts` - Explain highlighted text
- ✅ `src/services/DocumentParserWithMetadata.ts` - Track pages & sections

### Updated Components
- ✅ `src/components/ChatResponseDisplay.tsx` - NEW: Beautiful formatted responses with citations
- ✅ `src/components/ChatPanel.tsx` - Updated to use new response format
- ✅ `src/components/DocumentViewer.tsx` - NEW: Highlightable document with explain feature

### Updated ChatEngine
- ✅ `src/services/ChatEngine.ts` - Update query() to return CitationResponse
- ✅ `src/services/ChatEngine.ts` - Add compareDocuments() method
- ✅ `src/services/ChatEngine.ts` - Add smartSummarize() method
- ✅ `src/services/ChatEngine.ts` - Add explainHighlight() method

### Type Updates
- ✅ `src/types/workspace.ts` - Add Citation, ChatResponse types

## 9. Integration in ChatPanel

Example of how to use new services:

In [ ]:
// Updated ChatPanel.tsx - Example Integration
const handleSendMessage = async () => {
  if (!userQuery.trim()) return;

  setMessages(prev => [...prev, { 
    role: "user", 
    content: userQuery 
  }]);

  try {
    const chatEngine = ChatEngine.getInstance();
    const response = await chatEngine.query(userQuery);
    
    // Response now includes citations!
    setMessages(prev => [...prev, {
      role: "assistant",
      content: response.answer,
      data: response // Store full response with citations
    }]);
  } catch (error) {
    setMessages(prev => [...prev, {
      role: "assistant",
      content: `Error: ${error.message}`
    }]);
  }
};

// Compare documents
const handleCompare = async () => {
  const response = await MultiDocumentReasoningService
    .compareDocuments(
      "So sánh các tài liệu này",
      selectedDocuments.map(d => d.id),
      vectorStore
    );
  
  setMessages(prev => [...prev, {
    role: "assistant",
    content: response.answer,
    data: response
  }]);
};

// Highlight & Explain
const handleHighlight = async (selectedText: string) => {
  const response = await HighlightExplainService.explainHighlight(
    selectedText,
    currentDocument.id,
    documentContext,
    vectorStore
  );
  
  // Show explanation with citations
  setExplanationModal({
    open: true,
    response: response
  });
};

## Key Takeaways

✨ **NotebookLM-Style Features**
1. **Citations with Page Numbers** - "Theo trang 12: ..."
2. **Multi-Document Reasoning** - Compare 2+ docs intelligently
3. **Smart Summaries** - Full doc, section, or topic-specific
4. **Highlight & Explain** - Click text → AI explains with context
5. **Beautiful UI** - Professional response formatting with expandable citations

🎯 **Implementation Priority**
1. Add DocumentParserWithMetadata (page tracking)
2. Add CitationService (format citations)
3. Update ChatEngine.query() to return ChatResponse with citations
4. Create ChatResponseDisplay component (beautiful formatting)
5. Add remaining services (multi-doc, summarization, highlight)

💡 **Why This Makes It "Good"**
- Users know exactly where information comes from
- Can verify facts directly in source documents
- Professional looking like NotebookLM
- Increases trust in AI responses
- Enables multi-document workflows

# NotebookLM-Style Features Implementation Guide

Implementing advanced AI document analysis with:
1. **Source Citations** - Exact page/section references
2. **Multi-document Reasoning** - Compare & cross-reference documents
3. **Smart Summarization** - Full doc, section, or topic-specific
4. **Highlight & Explain** - Context-aware explanations
5. **Beautiful Response Formatting** - Professional presentation